# P11 · Calendario de medición de un indicador oficial

> **¿Qué días de 2026 entran en la medición según las reglas oficiales?**

Metodología DIPRES, listado versionado de feriados y calendario gregoriano.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/calendario_medicion/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# 1. Detectar el entorno y obtener una copia limpia del repositorio.
# Esta celda también funciona localmente si se abre el notebook dentro del repo.
import os
import platform
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
if not Path('pyproject.toml').exists():
    destino_repo = Path('lab-dtpm')
    if not destino_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git', str(destino_repo)], check=True)
    os.chdir(destino_repo)

ROOT = Path.cwd().resolve()
PAQUETES = ['pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'XlsxWriter', 'matplotlib', 'polars>=1.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES], check=True)
print('Entorno:', 'Google Colab' if EN_COLAB else 'Python local/Jupyter')
print('Python:', platform.python_version())
print('Repositorio:', ROOT)


In [ ]:
# 2. Importar y registrar versiones: evita resultados imposibles de auditar.
import matplotlib
import numpy as np
import pandas as pd
import polars as pl

print({'pandas': pd.__version__, 'polars': pl.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})


## Alternativas para integrar los datos

La vía reproducible es clonar GitHub y ejecutar `scripts/fetch_data.py`, como hace la siguiente celda. Si la red institucional lo impide, consulte la [guía portable](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/docs/ejecucion-portable.md): explica carga desde el equipo, montaje de Google Drive, descarga directa y validación SHA-256. No cambie la fuente por datos simulados sin declararlo.


In [ ]:
# 3. Obtener las fuentes originales declaradas por el caso.
# Si la fuente externa falla, el error se conserva: no se sustituyen datos
# reales por sintéticos. Los CSV versionados permiten ejecutar el laboratorio
# comparativo, pero no reemplazan la adquisición del caso completo.
fuentes = []
for fuente in fuentes:
    try:
        subprocess.run([sys.executable, 'scripts/fetch_data.py', fuente], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'No se pudo obtener {fuente}. Revise la URL, la red y las alternativas de carga descritas arriba.'
        ) from exc


## 1 · El problema

El denominador de un indicador oficial no es «los días del año». La metodología
descarta fines de semana, feriados, tres meses completos, vacaciones de invierno, la
semana de fiestas patrias y otras fechas. Además, la medición ocurre solo en cuatro
meses. Una nota al pie decide qué observaciones pueden entrar al indicador.

El riesgo no está en restar días, sino en sumar exclusiones solapadas. En 2026 las ocho
reglas marcan 256 días, pero el calendario contiene 365 y solo 201 quedan excluidos: 55
marcas se superponen. El cuaderno construye cada regla como una máscara auditable y
mide su aporte al denominador final.


## 2 · La solución que se va a aplicar

Se valida primero el listado de feriados contra Pascua y fechas fijas. Luego se genera
un calendario diario, se aplican las ocho reglas y se auditan sus solapamientos. El
resultado se cruza con los meses efectivos de medición. Por último se compara 2024–2027
para distinguir redundancias estructurales de coincidencias accidentales y se somete a
sensibilidad el único supuesto no fijado por la fuente: las vacaciones de invierno.


## Implementaciones comparables

La ruta principal conserva el caso completo en Pandas. El laboratorio comparable ejecuta Pandas y Polars en este runtime sobre el mismo CSV. R usa el mismo contrato y se valida en CI; el runtime Python estándar de Colab no ejecuta R de forma nativa.

#### Contrato común del ejercicio

- **Objetivo.** Comparar el denominador de días medibles entre años bajo las mismas reglas.
- **Fuente que leen las tres implementaciones.**
  [`resultados/comparacion_anios.csv`](resultados/comparacion_anios.csv), CSV UTF-8 versionado y producido
  por la resolución completa del caso. Los datos originales, su publicador, fecha,
  licencia, URL y huella se documentan en [`case.json`](case.json) y, cuando existe,
  `procedencia.json`. Este CSV es un **resultado reproducible**, no se presenta como
  una nueva fuente oficial.
- **Alcance.** días medibles por año. Los tres scripts comparables no sustituyen la adquisición,
  limpieza ni controles completos de los pasos Pandas: aíslan el indicador para que
  una diferencia observada sea atribuible al motor.
- **Fórmula o regla.** días_medibles_a = Σ 1(día laboral y no excluido por ninguna regla) para el año a.
- **Interpretación.** Las diferencias anuales pueden venir del calendario, aunque la metodología no cambie.
- **Error frecuente.** Reutilizar la lista de feriados de un año en otro altera el denominador silenciosamente.

| Campo | Tipo observado | Función en el ejercicio |
| --- | --- | --- |
| `anio` | `int64` | categoría/eje X |
| `dias_medibles` | `int64` | medida/eje Y |
| `fin_de_semana` | `int64` | campo auxiliar del contrato |
| `dias_28_y_29` | `int64` | campo auxiliar del contrato |
| `semana_fiestas_patrias` | `int64` | campo auxiliar del contrato |
| `feriados` | `int64` | campo auxiliar del contrato |
| `once_de_septiembre` | `int64` | campo auxiliar del contrato |
| `meses_completos` | `int64` | campo auxiliar del contrato |
| `vacaciones_invierno` | `int64` | campo auxiliar del contrato |
| `dia_joven_combatiente` | `int64` | campo auxiliar del contrato |

#### Qué cambia entre motores

| Dimensión | Pandas | Polars | R (`dplyr`/`readr`) |
| --- | --- | --- | --- |
| Modelo | DataFrame mutable por operaciones | expresiones columnares y plan optimizable | tubería declarativa sobre `tibble` |
| Nulos | `NaN`/`NA`; `groupby` exige decidir `dropna` | `null` separado de `NaN` | `NA`; muchas funciones exigen `na.rm` explícito |
| Orden | varias operaciones lo conservan, pero se ordena explícitamente | no se presupone tras agrupar | se fija con `arrange` |
| Gráfico | Matplotlib | Matplotlib tras extraer sólo X/Y | ggplot2 y su gramática de capas |
| Ventaja típica | ecosistema amplio y aprendizaje gradual | paralelismo, tipos y consultas grandes | estadística, comunicación y análisis reproducible |
| Límite aquí | el índice puede introducir alineaciones implícitas | añade una dependencia y otra semántica de tipos | requiere runtime y paquetes R independientes |

La equivalencia se valida sobre CSV normalizados con tolerancia numérica `1e-9`; no
se exige que la sintaxis ni los píxeles sean idénticos. Para ejecutar en un entorno
limpio consulte [`docs/ejecucion-portable.md`](../../docs/ejecucion-portable.md).

### Pandas + Matplotlib

```python
"""Laboratorio comparable Pandas + Matplotlib del caso calendario_medicion.

Lee el mismo contrato CSV que las versiones Polars y R. La resolución completa del
caso permanece en ``solution.py`` y en los pasos del cuaderno; este archivo aísla el
indicador central para que la comparación entre motores sea controlada.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Buscar la raíz desde el propio archivo evita rutas C:\... específicas del autor y
# funciona igual después de clonar el repositorio en Linux, Windows o Google Colab.
ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "calendario_medicion" / "resultados" / "comparacion_anios.csv"
REQUIRED = {"anio", "dias_medibles"}


def analizar() -> pd.DataFrame:
    """Carga, valida y transforma el indicador; no modifica el archivo de entrada."""
    datos = pd.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # La expresión conserva explícitos filtro, agrupación, orden y selección para que
    # el estudiante pueda alinearlos con las expresiones de Polars y dplyr.
    resultado = datos.sort_values("anio")[["anio", "dias_medibles"]]
    return resultado.reset_index(drop=True)


def graficar(resultado: pd.DataFrame, destino: Path) -> Path:
    """Representa exactamente las columnas comparadas y explicita sus unidades."""
    x = resultado["anio"].tolist()
    y = resultado["dias_medibles"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Año")
    ax.set_ylabel("Días medibles (recuento)")
    ax.set_ylim(bottom=0)
    ax.set_title("El denominador cambia con el calendario anual")
    ax.grid(axis="y", alpha=0.25)
    fig.text(0.01, 0.01, "Fuente: Resultados reproducibles del caso P11 · Período: 2024–2027", fontsize=8, color="#58677A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado.to_string(index=False))
    if args.salida:
        resultado.to_csv(args.salida, index=False)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `python cases/calendario_medicion/cuaderno/solucion_pandas.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/calendario_medicion/cuaderno/solucion_pandas.py', '--grafico', 'solucion_pandas-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_pandas-colab.png'))


### Polars + Matplotlib

```python
"""Laboratorio comparable Polars + Matplotlib del caso calendario_medicion.

Usa el mismo CSV, columnas, reglas y escala que Pandas y R. Polars expresa las
transformaciones como un plan de expresiones columnares; esa diferencia de método es
el objeto de aprendizaje, no una razón para cambiar la pregunta.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "calendario_medicion" / "resultados" / "comparacion_anios.csv"
REQUIRED = {"anio", "dias_medibles"}


def analizar() -> pl.DataFrame:
    """Carga y valida antes de transformar para producir un error explicativo."""
    datos = pl.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # select al final fija el contrato de salida y evita comparar columnas auxiliares.
    return datos.sort("anio").select(["anio", "dias_medibles"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    """Convierte sólo las series del gráfico; el análisis permanece en Polars."""
    x = resultado["anio"].to_list()
    y = resultado["dias_medibles"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Año")
    ax.set_ylabel("Días medibles (recuento)")
    ax.set_ylim(bottom=0)
    ax.set_title("El denominador cambia con el calendario anual")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P11 · Período: 2024–2027",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.salida:
        resultado.write_csv(args.salida)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/calendario_medicion/cuaderno/solucion_polars.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/calendario_medicion/cuaderno/solucion_polars.py', '--grafico', 'solucion_polars-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_polars-colab.png'))


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Laboratorio comparable R + dplyr/readr + ggplot2 del caso calendario_medicion.
#
# La resolución completa sigue en solution.py y en los pasos Pandas. Este script
# cambia el motor sobre el mismo contrato CSV para estudiar sintaxis y semántica sin
# cambiar datos, indicador ni escala gráfica.

required <- c("readr", "dplyr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  stop(
    "Faltan paquetes: ", paste(missing, collapse = ", "),
    ". Instálelos con install.packages(c('readr', 'dplyr', 'ggplot2'))."
  )
}

# Buscar pyproject.toml hacia arriba hace portable el script entre terminal, RStudio,
# Posit Cloud, IRkernel y un clon en Colab. LAB_ROOT permite una ruta explícita sin
# codificar la unidad o el directorio del desarrollador.
find_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "pyproject.toml"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("No se encontró la raíz del repositorio.")
    current <- parent
  }
}
root <- Sys.getenv("LAB_ROOT", unset = find_root())
source_csv <- file.path(root, "cases", "calendario_medicion", "resultados", "comparacion_anios.csv")

# readr conserva nombres y tipos previsibles. La validación temprana produce un error
# que explica el contrato roto, en vez de fallar más tarde dentro de una gráfica.
datos <- readr::read_csv(source_csv, show_col_types = FALSE, progress = FALSE)
required_columns <- c("anio", "dias_medibles")
missing_columns <- setdiff(required_columns, names(datos))
if (length(missing_columns)) stop("Faltan columnas: ", paste(missing_columns, collapse = ", "))

# dplyr expresa la misma selección, filtro, agrupación y orden que las otras vías.
resultado <- datos |> dplyr::arrange(.data$anio) |> dplyr::select(.data$anio, .data$dias_medibles)
print(resultado)

args <- commandArgs(trailingOnly = TRUE)
value_after <- function(flag) {
  pos <- match(flag, args)
  if (is.na(pos)) return(NULL)
  if (pos == length(args)) stop(flag, " necesita una ruta de salida")
  args[pos + 1]
}

csv_output <- value_after("--salida")
if (!is.null(csv_output)) readr::write_csv(resultado, csv_output)

plot_output <- value_after("--grafico")
if (!is.null(plot_output)) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["anio"]], y = .data[["dias_medibles"]])
  ) +
    ggplot2::geom_line(ggplot2::aes(group = 1), linewidth = 1, colour = "#11253B") +
    ggplot2::geom_point(size = 2.5, colour = "#55D6BE") +
    ggplot2::labs(
      title = "El denominador cambia con el calendario anual",
      x = "Año", y = "Días medibles (recuento)",
      caption = "Fuente: Resultados reproducibles del caso P11 · Período: 2024–2027"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(plot_output, grafico, width = 9, height = 5, dpi = 150)
  cat(plot_output, "\n")
}
```

Ejecutar con `Rscript cases/calendario_medicion/cuaderno/solucion.R`.


### Ejecutar R

Use `solucion.R` desde R/RStudio, Posit Cloud o Jupyter con IRkernel. También se incluye `solucion.Rmd`. En Colab, seleccione un runtime R compatible o use la instalación experimental documentada; se marca como compatibilidad documentada, no como ejecución validada.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.calendario_medicion import solution as p11

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Validar la fuente de feriados


El insumo externo se confronta con Pascua calculada y diez fechas fijas.

> ⚠️ Un CSV descargado no se vuelve verdadero por estar bien formado.


In [ ]:
resultados = p11.analizar(2026)
resultados['validacion_feriados']


### Paso 2 · Declarar las ocho reglas antes de calcular


Cada exclusión conserva nombre, descripción y fuente.

> ⚠️ Sumar los excluidos por regla cuenta 55 días más de una vez.


In [ ]:
resultados['exclusion_por_regla'][['regla','excluye','exclusiva','solapada','aporte_pct']]


### Paso 3 · Construir el calendario diario


Una fila por fecha permite auditar por qué cada día entra o sale.

**En Excel:** `=SECUENCIA(365;1;FECHA(2026;1;1);1)`

Debe dar: **365 fechas**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Trabajar solo con totales impide explicar un día particular.


In [ ]:
resultados['resumen_calendario']


### Paso 4 · Medir solapamientos y redundancias


Una regla puede excluir días y aun no aportar ninguno dado el resto.

> ⚠️ Que una regla sea redundante en un año no demuestra que sea innecesaria siempre.


In [ ]:
resultados['reglas_redundantes']


### Paso 5 · Cruzar con los meses efectivos de medición


El denominador real es la intersección, no todos los días elegibles del año.

**En Excel:** `=Y(C2;ESNUMERO(COINCIDIR(MES(A2);{3;6;9;11};0)))`

Debe dar: **78 días medibles**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Publicar 164 días elegibles responde otra pregunta; solo 78 están en meses de medición.


In [ ]:
resultados['resumen_medicion']


### Paso 6 · Cuantificar el aporte al denominador final


Se quita una regla a la vez y se recalcula la intersección.

> ⚠️ Meses completos excluye 90 días del año y cero días del denominador medido.


In [ ]:
resultados['aporte_sobre_medibles']


### Paso 7 · Distinguir redundancia estructural de accidental


Cuatro años muestran qué reglas jamás afectan y cuáles dependen del calendario.

> ⚠️ Feriados no afecta en 2026, pero sí elimina entre uno y tres días en otros años.


In [ ]:
resultados['clasificacion_reglas']


### Paso 8 · Medir el supuesto de vacaciones de invierno


La fecha no publicada se desplaza ±14 días para comprobar si cambia la decisión.

> ⚠️ Mover el supuesto a junio reduce el denominador hasta seis días; no es un detalle narrativo.


In [ ]:
resultados['sensibilidad_invierno']


## 4 · El resultado

De 261 días laborales quedan 164 elegibles y 78 medibles. Cuatro reglas no cambian el denominador de 2026, pero solo tres son estructuralmente redundantes en 2024–2027.

## 5 · Qué queda aprendido

1. El denominador es una decisión documentada y debe poder explicarse fecha por fecha.
2. Las exclusiones se combinan como conjuntos; sumar conteos ignora sus intersecciones.
3. Una redundancia de un año puede ser accidental.
4. Todo supuesto de calendario se somete a sensibilidad antes de publicar.
